In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ar = np.array([3,5,6,2,1,5])


In [ ]:
np.convolve(ar, [1,2,3]) # (3,2,1)

array([ 3, 11, 25, 29, 23, 13, 13, 15])

**Covolution On the Images from load_digits**

In [ ]:
from sklearn.datasets import load_digits

In [ ]:
dt = load_digits(n_class=3)

In [ ]:
dt.keys()

dict_keys(['data', 'target', 'frame', 'feature_names', 'target_names', 'images', 'DESCR'])

In [ ]:
dt.images[0]

array([[ 0.,  0.,  5., 13.,  9.,  1.,  0.,  0.],
       [ 0.,  0., 13., 15., 10., 15.,  5.,  0.],
       [ 0.,  3., 15.,  2.,  0., 11.,  8.,  0.],
       [ 0.,  4., 12.,  0.,  0.,  8.,  8.,  0.],
       [ 0.,  5.,  8.,  0.,  0.,  9.,  8.,  0.],
       [ 0.,  4., 11.,  0.,  1., 12.,  7.,  0.],
       [ 0.,  2., 14.,  5., 10., 12.,  0.,  0.],
       [ 0.,  0.,  6., 13., 10.,  0.,  0.,  0.]])

In [ ]:
# dt.target

In [ ]:
dt.images.shape

(537, 8, 8)

In [ ]:
X = dt.images
Y = dt.target

In [ ]:
X = X.reshape(537,1,8,8)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam

In [ ]:
X = torch.FloatTensor(X)
Y = torch.LongTensor(Y)

In [ ]:
X.shape

torch.Size([537, 1, 8, 8])

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(64*2*2, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
)

In [ ]:
loss = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr=0.001)

In [ ]:
c=0
for epoch in range(100):
    c=c+1
    opt.zero_grad()
    yp = model(X)
    ls = loss(yp, Y)
    ls.backward()
    opt.step()
    if c%10==0:
      print(ls.item())

0.09067617356777191
0.010078852996230125
0.002396010560914874
0.0008692945120856166
0.0004664821026381105
0.0003162780194543302
0.00023496906214859337
0.00018544623162597418
0.00015191490820143372
0.00012707144196610898


In [ ]:
# Prediction on new data

In [ ]:
newx = X[112].reshape(1,1,8,8)

In [ ]:
newx.shape

torch.Size([1, 1, 8, 8])

In [ ]:
Yp = model(newx)

In [ ]:
Yp

tensor([[-3.1934,  4.5609, -4.9166]], grad_fn=<AddmmBackward0>)

In [ ]:
sx = torch.softmax(Yp, dim=1)

In [ ]:
torch.argmax(sx, dim=1)

tensor([1])

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Load data
dt = load_digits(n_class=3)
X = dt.images
Y = dt.target

X = X.reshape(-1, 1, 8, 8)

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Convert to tensors
X_train = torch.FloatTensor(X_train)
Y_train = torch.LongTensor(Y_train)
X_test = torch.FloatTensor(X_test)
Y_test = torch.LongTensor(Y_test)

# Datasets and loaders
train_dataset = TensorDataset(X_train, Y_train)
test_dataset = TensorDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    nn.Flatten(),
    nn.Linear(64 * 2 * 2, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
)

loss_fn = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr=0.001)

# Training
for epoch in range(20):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        opt.zero_grad()
        yp = model(xb)
        ls = loss_fn(yp, yb)
        ls.backward()
        opt.step()
        total_loss += ls.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        yp = model(xb)
        pred = torch.argmax(yp, dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)

print("Test Accuracy:", correct / total)

Epoch 1, Loss: 7.6323
Epoch 2, Loss: 0.8852
Epoch 3, Loss: 0.2664
Epoch 4, Loss: 0.0640
Epoch 5, Loss: 0.0340
Epoch 6, Loss: 0.0212
Epoch 7, Loss: 0.0196
Epoch 8, Loss: 0.0161
Epoch 9, Loss: 0.0125
Epoch 10, Loss: 0.0113
Epoch 11, Loss: 0.0094
Epoch 12, Loss: 0.0089
Epoch 13, Loss: 0.0073
Epoch 14, Loss: 0.0071
Epoch 15, Loss: 0.0061
Epoch 16, Loss: 0.0054
Epoch 17, Loss: 0.0057
Epoch 18, Loss: 0.0048
Epoch 19, Loss: 0.0040
Epoch 20, Loss: 0.0037
Test Accuracy: 1.0


# Class based Model

In [ ]:
class MyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
    self.maxm = nn.MaxPool2d(kernel_size=2, stride=2)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    self.relu = nn.ReLU()
    self.flatten = nn.Flatten()
    self.linear1 = nn.Linear(64*2*2, 128)
    self.linear2 = nn.Linear(128, 3)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.flatten(x)
    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)
    return x

In [ ]:
class MyNet(nn.Module):
  def __init__(sf):
    super().__init__()
    sf.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
    sf.maxm = nn.MaxPool2d(kernel_size=2, stride=2)
    sf.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    sf.relu = nn.ReLU()
    sf.flatten = nn.Flatten()
    sf.linear1 = nn.Linear(64*2*2, 128)
    sf.linear2 = nn.Linear(128, 3)

  def forward(sf, x):
    print(x.shape)
    x = sf.conv1(x)
    print(x.shape)
    x = sf.relu(x)
    x = sf.maxm(x)
    x = sf.conv2(x)
    x = sf.relu(x)
    x = sf.maxm(x)
    x = sf.flatten(x)
    x = sf.linear1(x)
    x = sf.relu(x)
    x = sf.linear2(x)
    return x

In [ ]:
mod = MyNet()

In [ ]:
mod(X)

In [ ]:
loss = nn.CrossEntropyLoss()
opt = Adam(mod.parameters(), lr=0.001)

In [ ]:
c=0
for epoch in range(100):
    c=c+1
    opt.zero_grad()
    yp = mod(X)
    ls = loss(yp, Y)
    ls.backward()
    opt.step()
    if c%10==0:
      print(ls.item())

0.11996563524007797
0.008261230774223804
0.001587284030392766
0.0006886046612635255
0.0003927003708668053
0.00029141659615561366
0.0002355073083890602
0.000198277150047943
0.00017504052084404975
0.00015494856052100658


# Glimpse of Transfer Learning

### How to disable some of gradients from getting changed

In [ ]:
c=0
for p in model.parameters():
  c=c+1
  if c==7:
    p.requires_grad = True
  else:
    p.requires_grad = False



In [ ]:
model

Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=256, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=3, bias=True)
)

In [ ]:
# model(X)

In [ ]:
ls[0].shape

torch.Size([32, 1, 3, 3])

In [ ]:
ls[2].shape

torch.Size([64, 32, 3, 3])

In [ ]:
ls = []
for p in model.parameters():
  if p.requires_grad:
    ls.append(p.shape)


torch.Size([32, 1, 3, 3])
torch.Size([32])
torch.Size([64, 32, 3, 3])
torch.Size([64])
torch.Size([128, 256])
torch.Size([128])
torch.Size([3, 128])
torch.Size([3])
